# Clasificación de Gestos EMG con Redes Neuronales Híbridas
### Trabajo de Grado — Escuela de Ingeniería Eléctrica y Electrónica, Universidad del Valle
**Dataset:** NinaPro DB3 · **Sujeto evaluado:** S1 · **Objetivo:** > 79.72% (baseline del paper)

---
Este notebook demuestra el pipeline completo: datos crudos → preprocesamiento → modelo → resultados.

In [ ]:
import os
# Asegura que el directorio de trabajo sea la raíz del proyecto
if os.path.basename(os.getcwd()) != 'Clasificador_SEMG':
    os.chdir('..')  # Si se abre desde una subcarpeta
print(f'Directorio: {os.getcwd()}')

import torch
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import butter, filtfilt
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from pathlib import Path
from tqdm.notebook import tqdm

# Configuración de gráficas
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Dispositivo: {DEVICE}')

## 1. Datos crudos — NinaPro DB3
La señal EMG fue capturada con 12 electrodos de superficie (Delsys Trigno) a **2000 Hz**.
Cada sujeto realizó **6 repeticiones** de 7 gestos: reposo + 6 flexiones de dedos.

In [ ]:
# Cargar archivo .mat de S1
mat = scipy.io.loadmat('data/raw/S1_E1_A1.mat')
emg_raw   = mat['emg']          # (N, 12)
stimulus  = mat['restimulus'].flatten()
repetition = mat['rerepetition'].flatten()

print(f'Señal EMG cruda:  {emg_raw.shape}  ({emg_raw.shape[0]/2000:.0f} s a 2000 Hz)')
print(f'Canales EMG:      {emg_raw.shape[1]}')
print(f'Clases únicas:    {np.unique(stimulus)}')
print(f'Repeticiones:     {np.unique(repetition)}')

GESTOS = ['Reposo', 'Gesto 1\n(flexión\npulgar)', 'Gesto 2\n(flexión\níndice)',
          'Gesto 3\n(flexión\nmedio)', 'Gesto 4\n(flexión\nanular)',
          'Gesto 5\n(flexión\nmeñique)', 'Gesto 6\n(flexión\ncuatro dedos)']
GESTOS_CORTO = ['Reposo', 'G1', 'G2', 'G3', 'G4', 'G5', 'G6']
COLORES = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4', '#FF5722']

In [ ]:
# Visualizar distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Segmento de señal cruda (5 segundos)
t_inicio, t_fin = 80*2000, 85*2000  # segundos 80-85
t = np.arange(t_inicio, t_fin) / 2000
ax = axes[0]
for ch in range(0, 12, 2):  # cada 2 canales para claridad
    offset = ch * 0.001
    ax.plot(t, emg_raw[t_inicio:t_fin, ch] + offset, lw=0.5, alpha=0.8, label=f'Ch{ch+1}')
stim_seg = stimulus[t_inicio:t_fin]
for g in range(1, 7):
    mask = stim_seg == g
    if mask.any():
        ax.axvspan(t[mask][0], t[mask][-1], alpha=0.15, color=COLORES[g], label=GESTOS_CORTO[g])
ax.set_xlabel('Tiempo (s)')
ax.set_ylabel('Amplitud (V)')
ax.set_title('Señal EMG cruda — S1 (5 segundos)')
ax.legend(loc='upper right', fontsize=8, ncol=2)

# Distribución de clases
ax2 = axes[1]
counts = [np.sum(stimulus == g) / 2000 for g in range(7)]  # en segundos
bars = ax2.bar(GESTOS_CORTO, counts, color=COLORES, edgecolor='white', linewidth=0.5)
ax2.set_ylabel('Duración total (s)')
ax2.set_title('Distribución de clases — S1 (señal completa)')
for bar, c in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{c:.0f}s', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('checkpoints/fig_raw_data.png', bbox_inches='tight')
plt.show()

## 2. Pipeline de preprocesamiento
Cada señal pasa por: **Downsample** (2000→500 Hz) → **Bandpass** (20–249 Hz) → **Z-score** (normalización por canal)

In [ ]:
# Seleccionar un segmento de gesto para demostrar preprocesamiento
TARGET_FREQUENCY = 500
DOWNSAMPLING_FACTOR = 2000 // TARGET_FREQUENCY

# Encontrar primer gesto 1 (rep 1)
mask_g1_r1 = (stimulus == 1) & (repetition == 1)
idx_g1 = np.where(mask_g1_r1)[0]
seg_raw = emg_raw[idx_g1[0]-500 : idx_g1[-1]+500, 0]  # canal 0, con contexto

# 1. Downsample
seg_ds = seg_raw[::DOWNSAMPLING_FACTOR]

# 2. Bandpass 20-249 Hz
nyq = TARGET_FREQUENCY / 2
b, a = butter(4, [20/nyq, 249/nyq], btype='band')
seg_bp = filtfilt(b, a, seg_ds)

# 3. Z-score
seg_zs = (seg_bp - seg_bp.mean()) / (seg_bp.std() + 1e-8)

# Visualizar pipeline
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=False)
pasos = [
    (np.arange(len(seg_raw))/2000, seg_raw, '2000 Hz', 'Cruda — 2000 Hz'),
    (np.arange(len(seg_ds))/500,   seg_ds,  '500 Hz',  'Downsample → 500 Hz'),
    (np.arange(len(seg_zs))/500,   seg_zs,  'Z-score', 'Bandpass 20-249 Hz + Z-score'),
]
for ax, (t_arr, s, ylabel, titulo) in zip(axes, pasos):
    ax.plot(t_arr, s, lw=0.7, color='#1565C0')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(titulo, fontsize=11, pad=3)
    ax.grid(alpha=0.3)
    ax.axvspan(idx_g1[0]/2000, idx_g1[-1]/2000, alpha=0.1, color='green', label='Gesto 1')

axes[-1].set_xlabel('Tiempo (s)')
plt.suptitle('Pipeline de preprocesamiento — Canal 1, Sujeto S1', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('checkpoints/fig_preprocessing.png', bbox_inches='tight')
plt.show()

## 3. Ventaneo y Features Manuales
La señal se divide en **ventanas de 250 ms (125 muestras)** con stride de **50 ms (25 muestras)**.  
Cada ventana genera dos entradas al modelo:
- **Señal:** tensor `(1, 125, 13)` — 12 canales EMG + 1 canal RMS
- **Features manuales:** vector `(48,)` — RMS, ZC, SSC, WL por cada canal

In [ ]:
WINDOW_SIZE = 125
STRIDE = 25

# Cargar datos procesados usando el data_loader real
from src.data_loader import create_data_loaders

_, _, test_loader = create_data_loaders(batch_size=64)
print(f'Muestras de test (rep 6): {len(test_loader.dataset)}')

# Tomar una muestra para visualizar
signals_batch, features_batch, labels_batch = next(iter(test_loader))
idx_demo = 10  # índice arbitrario en el batch

sig  = signals_batch[idx_demo, 0].numpy()   # (125, 13)
feat = features_batch[idx_demo].numpy()     # (48,)
lbl  = labels_batch[idx_demo].item()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Señal multicanal
ax = axes[0]
t_win = np.arange(WINDOW_SIZE) / TARGET_FREQUENCY * 1000  # en ms
for ch in range(12):
    offset = ch * 3
    ax.plot(t_win, sig[:, ch] + offset, lw=0.8, alpha=0.9)
# Canal RMS en rojo
ax.plot(t_win, sig[:, 12] * 3 - 3, lw=1.2, color='red', alpha=0.8, label='RMS (ch 13)')
ax.set_xlabel('Tiempo (ms)')
ax.set_ylabel('Canales (con offset)')
ax.set_title(f'Ventana 250ms — Clase: {GESTOS_CORTO[lbl]}')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Features manuales
ax2 = axes[1]
feat_names = ['RMS', 'ZC', 'SSC', 'WL']
x = np.arange(12)
width = 0.2
colors_feat = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
for i, (name, col) in enumerate(zip(feat_names, colors_feat)):
    vals = feat[i::4]  # features de tipo i para los 12 canales
    # Normalizar para visualizar juntos
    vn = (vals - vals.min()) / (vals.max() - vals.min() + 1e-8)
    ax2.bar(x + i*width, vn, width=width, label=name, color=col, alpha=0.8)
ax2.set_xlabel('Canal EMG')
ax2.set_ylabel('Valor normalizado')
ax2.set_title('Features manuales (48D) — normalizados para comparar')
ax2.set_xticks(x + 1.5*width)
ax2.set_xticklabels([f'Ch{i+1}' for i in range(12)], fontsize=8)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('checkpoints/fig_window_features.png', bbox_inches='tight')
plt.show()

## 4. Arquitectura CyborgHybrid (v10)
La arquitectura combina dos ramas paralelas que procesan la misma señal de maneras complementarias.

In [ ]:
from src.models_cyborg import CyborgHybrid

model_v10 = CyborgHybrid(manual_features_dim=48)
model_v10.get_model_info()

# Tabla de parámetros por componente
total = sum(p.numel() for p in model_v10.parameters())
componentes = {
    'CNN (Conv1D ×3)': sum(p.numel() for n,p in model_v10.named_parameters() if 'conv' in n),
    'BiLSTM':          sum(p.numel() for n,p in model_v10.named_parameters() if 'lstm' in n),
    'MLP features':    sum(p.numel() for n,p in model_v10.named_parameters() if 'feature_mlp' in n),
    'Clasificador':    sum(p.numel() for n,p in model_v10.named_parameters() if 'classifier' in n),
}

fig, ax = plt.subplots(figsize=(8, 4))
names  = list(componentes.keys())
params = list(componentes.values())
cols   = ['#1565C0', '#283593', '#2E7D32', '#6A1B9A']
bars   = ax.barh(names, [p/1000 for p in params], color=cols, edgecolor='white')
for bar, p in zip(bars, params):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{p:,} ({p/total*100:.1f}%)', va='center', fontsize=10)
ax.set_xlabel('Parámetros (miles)')
ax.set_title(f'Distribución de parámetros — CyborgHybrid ({total:,} total)')
ax.set_xlim(0, max(p/1000 for p in params) * 1.35)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('checkpoints/fig_params.png', bbox_inches='tight')
plt.show()

## 5. Cargar el Ensemble 4M (mejor resultado: 72.88%)
El ensemble combina 4 modelos con representaciones complementarias.

In [ ]:
from src.models_v18 import CyborgHybridAttention
from src.train_v17_features import create_data_loaders_v17

CHECKPOINTS = Path('checkpoints')

def load_model(ModelClass, ckpt_path, features_dim=None, device='cpu'):
    if features_dim is not None:
        model = ModelClass(manual_features_dim=features_dim)
    else:
        model = ModelClass()
    ckpt = torch.load(ckpt_path, weights_only=False, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval().to(device)
    val_acc = ckpt.get('val_accuracy', 0)
    return model, val_acc

print('Cargando modelos del Ensemble 4M...')
m10, v10_val = load_model(CyborgHybrid,           CHECKPOINTS/'best_model_cyborg.pth',         48,  DEVICE)
m15, v15_val = load_model(CyborgHybrid,           CHECKPOINTS/'best_model_v15_augmentation.pth', 48,  DEVICE)
m17, v17_val = load_model(CyborgHybrid,           CHECKPOINTS/'best_model_v17_features.pth',    132, DEVICE)
m18, v18_val = load_model(CyborgHybridAttention,  CHECKPOINTS/'best_model_v18_attention.pth',   None, DEVICE)

_, _, test_48  = create_data_loaders(batch_size=64)
_, _, test_132 = create_data_loaders_v17(batch_size=64)

info_modelos = [
    ('v10  Cyborg base',       f'{v10_val*100:.1f}%', '48D',  'CNN+BiLSTM clásico'),
    ('v15  + Augmentation',    f'{v15_val*100:.1f}%', '48D',  'Ruido+scaling+dropout de canales'),
    ('v17  Features 132D',     f'{v17_val*100:.1f}%', '132D', 'MAV, VAR, WAMP, AR(1-4) añadidos'),
    ('v18  Self-Attention',    f'{v18_val*100:.1f}%', '48D',  'MHA 8 heads sobre salida BiLSTM'),
]
print(f'\n{"Modelo":<25} {"Val Acc":>8} {"Features":>8}  Descripción')
print('-' * 70)
for nombre, val, feat, desc in info_modelos:
    print(f'{nombre:<25} {val:>8} {feat:>8}  {desc}')

## 6. Inferencia del Ensemble en el Test Set

In [ ]:
def get_probs(model, loader, device):
    """Obtiene probabilidades softmax del modelo sobre el loader."""
    all_probs, all_labels = [], []
    with torch.no_grad():
        for signals, features, labels in tqdm(loader, leave=False):
            logits = model(signals.to(device), features.to(device))
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.concatenate(all_probs, axis=0), np.array(all_labels)

print('Calculando predicciones...')
p10, y_true = get_probs(m10, test_48,  DEVICE)
p15, _      = get_probs(m15, test_48,  DEVICE)
p17, _      = get_probs(m17, test_132, DEVICE)
p18, _      = get_probs(m18, test_48,  DEVICE)

# Ensemble: promedio uniforme
p_ensemble = (p10 + p15 + p17 + p18) / 4
y_pred_ens = np.argmax(p_ensemble, axis=1)
y_pred_v10 = np.argmax(p10, axis=1)

acc_v10 = np.mean(y_pred_v10 == y_true)
acc_ens = np.mean(y_pred_ens == y_true)
bal_v10 = balanced_accuracy_score(y_true, y_pred_v10)
bal_ens = balanced_accuracy_score(y_true, y_pred_ens)

print(f'\nv10 individual:  Acc={acc_v10*100:.2f}%  BalAcc={bal_v10*100:.2f}%')
print(f'Ensemble 4M:     Acc={acc_ens*100:.2f}%  BalAcc={bal_ens*100:.2f}%')
print(f'Mejora ensemble: +{(acc_ens-acc_v10)*100:.2f}%')

## 7. Matriz de Confusión — Ensemble 4M

In [ ]:
cm = confusion_matrix(y_true, y_pred_ens)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # normalizada por fila

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mat, titulo, fmt, vmax in [
    (axes[0], cm_norm, 'Matriz de Confusión Normalizada (Ensemble 4M)', '.2f', 1.0),
    (axes[1], cm,      'Conteos absolutos',                              'd',   None),
]:
    im = ax.imshow(mat, interpolation='nearest', cmap='Blues', vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, shrink=0.85)
    ax.set_xticks(range(7))
    ax.set_yticks(range(7))
    ax.set_xticklabels(GESTOS_CORTO, fontsize=10)
    ax.set_yticklabels(GESTOS_CORTO, fontsize=10)
    ax.set_xlabel('Predicho', fontsize=11)
    ax.set_ylabel('Real', fontsize=11)
    ax.set_title(titulo, fontsize=12)
    thresh = mat.max() / 2.
    for i in range(7):
        for j in range(7):
            val = mat[i, j]
            ax.text(j, i, format(val, fmt), ha='center', va='center',
                    fontsize=9, color='white' if val > thresh else 'black')

plt.suptitle(f'Ensemble 4M — Test Accuracy: {acc_ens*100:.2f}%  |  BalAcc: {bal_ens*100:.2f}%',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('checkpoints/fig_confusion_matrix.png', bbox_inches='tight')
plt.show()

## 8. Accuracy por clase — Ensemble vs modelos individuales

In [ ]:
def per_class_acc(y_true, y_pred):
    cm_ = confusion_matrix(y_true, y_pred)
    return [cm_[i,i]/cm_[i].sum() if cm_[i].sum()>0 else 0 for i in range(7)]

acc_por_clase = {
    'v10 (base)':    per_class_acc(y_true, y_pred_v10),
    'Ensemble 4M':   per_class_acc(y_true, y_pred_ens),
}

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(7)
width = 0.35

bars1 = ax.bar(x - width/2, [v*100 for v in acc_por_clase['v10 (base)']],
               width, label='v10 individual', color='#90CAF9', edgecolor='#1565C0')
bars2 = ax.bar(x + width/2, [v*100 for v in acc_por_clase['Ensemble 4M']],
               width, label='Ensemble 4M', color='#1565C0', edgecolor='#0D47A1')

# Línea del objetivo
ax.axhline(79.72, color='red', linestyle='--', linewidth=1.5, label='Objetivo paper (79.72%)')
ax.axhline(100/7, color='gray', linestyle=':', linewidth=1, label='Azar (14.3%)')

# Anotar valores
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.0f}%',
                ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(GESTOS_CORTO, fontsize=11)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy por clase: v10 individual vs Ensemble 4M')
ax.set_ylim(0, 115)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('checkpoints/fig_per_class.png', bbox_inches='tight')
plt.show()

## 9. Evolución de resultados — todas las versiones

In [ ]:
resultados = [
    ('v7  CNN+BiLSTM',          69.95, False),
    ('v10 CyborgHybrid',        71.32, False),
    ('v11 Focal (agresivo)',     51.99, True),
    ('v12 Focal (moderado)',     70.17, False),
    ('Cascada HMM',             67.45, True),
    ('v13 Multi-sujeto',        68.91, True),
    ('v14 Fine-tuning',         70.69, False),
    ('v15 Augmentation',        70.74, False),
    ('Ensemble 2M (v10+v15)',   72.41, False),
    ('v17 Features 132D',       71.89, False),
    ('Ensemble 3M pond.',       72.57, False),
    ('v18 Self-Attention',      71.63, False),
    ('Ensemble 4M ★',          72.88, False),
    ('v19 TCN',                 69.80, True),
    ('v19b TCN+Cosine',         70.38, True),
    ('v20 Spectral 108D',       70.11, True),
    ('v21 Combined 192D',       70.32, True),
    ('Cascada v22',             70.27, True),
    ('v23 Transformer',         71.89, False),
    ('v24 Conformer',           71.00, True),
    ('Ensemble 5M+v23',         72.78, False),
    ('v25 STFT+CNN2D',          68.70, True),
    ('v26 LabelSmooth+SWA',     71.47, False),
]

nombres = [r[0] for r in resultados]
valores = [r[1] for r in resultados]
descartados = [r[2] for r in resultados]

fig, ax = plt.subplots(figsize=(14, 8))
colores_bar = ['#EF9A9A' if d else '#1565C0' for d in descartados]
# Destacar el mejor
colores_bar[nombres.index('Ensemble 4M ★')] = '#F9A825'

bars = ax.barh(nombres, valores, color=colores_bar, edgecolor='white', height=0.7)

# Líneas de referencia
ax.axvline(79.72, color='red',   linestyle='--', lw=2, label='Objetivo paper: 79.72%')
ax.axvline(72.88, color='#F9A825', linestyle='--', lw=1.5, label='Máximo actual: 72.88%')
ax.axvline(71.32, color='gray',  linestyle=':', lw=1, label='v10 referencia: 71.32%')

# Anotar valores
for bar, val in zip(bars, valores):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=8)

# Leyenda de colores
legend_patches = [
    mpatches.Patch(color='#1565C0', label='Modelo activo / competitivo'),
    mpatches.Patch(color='#EF9A9A', label='Estrategia descartada'),
    mpatches.Patch(color='#F9A825', label='Mejor resultado'),
]
ax.legend(handles=legend_patches + [
    plt.Line2D([0], [0], color='red',    lw=2, linestyle='--', label='Objetivo: 79.72%'),
    plt.Line2D([0], [0], color='#F9A825',lw=1.5, linestyle='--', label='Máximo: 72.88%'),
], fontsize=9, loc='lower right')

ax.set_xlabel('Test Accuracy (%)')
ax.set_title('Evolución de resultados — 23 versiones y estrategias evaluadas', fontsize=13)
ax.set_xlim(48, 83)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('checkpoints/fig_evolution.png', bbox_inches='tight')
plt.show()

## 10. Demostración de inferencia sobre una ventana

In [ ]:
# Elegir ventanas de diferentes clases para demostrar
fig, axes = plt.subplots(1, 7, figsize=(16, 4))

# Acumular una muestra por clase
ejemplos = {}
for signals, features, labels in test_loader:
    for s, f, l in zip(signals, features, labels):
        g = l.item()
        if g not in ejemplos:
            ejemplos[g] = (s.unsqueeze(0), f.unsqueeze(0))
    if len(ejemplos) == 7:
        break

for g in range(7):
    sig_g, feat_g = ejemplos[g]
    with torch.no_grad():
        sig_dev = sig_g.to(DEVICE); feat_dev = feat_g.to(DEVICE)
        p_ens_g = sum([
            torch.softmax(m(sig_dev, feat_dev), dim=1).cpu().numpy()
            for m, f_loader in [
                (m10, feat_g), (m15, feat_g),
            ]
        ])
        p10_g = torch.softmax(m10(sig_dev, feat_dev), dim=1).cpu().numpy()[0]
        p15_g = torch.softmax(m15(sig_dev, feat_dev), dim=1).cpu().numpy()[0]
        # v17 y v18 usan el mismo loader_48 para simplificar esta celda
        p_g   = (p10_g + p15_g) / 2  # simplificado: 2M
    
    pred  = p_g.argmax()
    correcto = (pred == g)
    
    ax = axes[g]
    bars = ax.bar(range(7), p_g * 100, color=['#EF9A9A']*7)
    bars[pred].set_color('#1565C0' if correcto else '#E53935')
    bars[g].set_edgecolor('#F9A825'); bars[g].set_linewidth(2)
    ax.set_title(f'{GESTOS_CORTO[g]}\n{"✓" if correcto else "✗"} → {GESTOS_CORTO[pred]}',
                 fontsize=10, color='#2E7D32' if correcto else '#C62828')
    ax.set_xticks(range(7))
    ax.set_xticklabels(GESTOS_CORTO, fontsize=7)
    ax.set_ylim(0, 105)
    ax.set_ylabel('Confianza (%)' if g == 0 else '')
    ax.grid(alpha=0.3, axis='y')

plt.suptitle('Distribución de confianza por clase — una ventana de ejemplo por gesto',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('checkpoints/fig_confidence_demo.png', bbox_inches='tight')
plt.show()

## 11. Resumen final

In [ ]:
bal_ens = balanced_accuracy_score(y_true, y_pred_ens)
cm_ens  = confusion_matrix(y_true, y_pred_ens)

print('=' * 60)
print('RESUMEN — TRABAJO DE GRADO')
print('Clasificación de Gestos EMG con Redes Neuronales Híbridas')
print('=' * 60)
print(f'\nDataset:           NinaPro DB3 (S1, rep 6 = test set)')
print(f'Muestras de test:  {len(y_true)} ventanas de 250ms')
print(f'Clases:            7 (reposo + 6 gestos)')
print()
print(f'MEJOR RESULTADO:   Ensemble 4M (v10+v15+v17+v18)')
print(f'  Test Accuracy:   {acc_ens*100:.2f}%')
print(f'  Balanced Acc:   {bal_ens*100:.2f}%')
print(f'  Objetivo paper: 79.72%  (gap: {(79.72-acc_ens*100):.2f}%)')
print()
print('Accuracy por clase (Ensemble 4M):')
for i in range(7):
    acc_i = cm_ens[i,i]/cm_ens[i].sum() * 100
    bar = '█' * int(acc_i/5)
    print(f'  {GESTOS_CORTO[i]:>6}: {acc_i:5.1f}%  {bar}')
print()
print('Versiones evaluadas: 23')
print('Estrategias exitosas: Ensemble de modelos diversos')
print('Checkpoints guardados:', list(Path('checkpoints').glob('fig_*.png')))
print('=' * 60)